# QC Anomaly Detection - Interactive Analysis Notebook

This notebook demonstrates the complete workflow for QC anomaly detection:
1. Load data
2. Define filters
3. Generate filtered dataset
4. Run anomaly detection
5. Visualize results

**Note**: This is a simulation of the final system. It is used for testing purpose during build

## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import sys

## 1. test Data loading scripts/functions


In [2]:
from src.data_validator import validate_structure, validate_lcs_data

# Ensure the project root is on the path so src.* imports work
sys.path.insert(0, str(Path(".").resolve()))

data_filepath = r"data\raw\QC_Sample_Data.csv"  # UPDATE IF NEEDED

# Load data — prints column list, type/status breakdown, and validation summary
df = pd.read_csv(data_filepath)
# ----------------------------------------------------------------
# Test 1: validate_structure — every source_column in column_config.csv
# should be found in the loaded df.
# ----------------------------------------------------------------
print("TEST 1: validate_structure(df)")
print("=" * 60)
structure_ok = validate_structure(df, config_dir="config/")
print(f"\n>> validate_structure returned: {structure_ok}")

# ----------------------------------------------------------------
# Test 2: validate_lcs_data — Control/LCS subset should have no
# missing values or dtype mismatches in the columns LCSDetector needs.
# ----------------------------------------------------------------
print("\n\nTEST 2: validate_lcs_data(df)")
print("=" * 60)
lcs_validation = validate_lcs_data(df)
print(f"\n>> validate_lcs_data returned:")
lcs_validation 

TEST 1: validate_structure(df)
[DataValidator] -- Structure validation ----------------------------------------
    [ok]  ANALYTICAL_TYPE
    [!]   MISSING (optional)  QC_TYPE
    [!]   MISSING (optional)  STD_LOT_CODE
    [!]   MISSING (optional)  STD_CODE
    [!]   MISSING (optional)  JOB_CODE
    [ok]  NUMERIC_FINAL_VALUE
    [ok]  ANALYSED_DATE
    [ok]  SCHEME_CODE
    [ok]  ANALYTE_CODE
    [ok]  STANDARD_STATUS
    [ok]  PRECISION_STATUS
    [!]   MISSING (optional)  INTERNAL_MIN_VALUE
    [!]   MISSING (optional)  INTERNAL_MAX_VALUE
    [!]   MISSING (optional)  INTERNAL_MIN_INCLUSIVE
    [!]   MISSING (optional)  INTERNAL_MAX_INCLUSIVE
    [!]   MISSING (optional)  INTERNAL_MAX_WARNING_VALUE
    [!]   MISSING (optional)  INTERNAL_MIN_WARNING_VALUE
    [!]   MISSING (optional)  INTERNAL_MIN_WARNING_INCLUSIVE
    [!]   MISSING (optional)  INTERNAL_MAX_WARNING_INCLUSIVE
    [!]   MISSING (optional)  LIM_REP_VALUE
    [!]   MISSING (optional)  STAT_DL_VALUE
    [!]   MISSING (opti

{'status': False,
 'n_rows': 0,
 'missing_columns': ['STD_LOT_CODE',
  'STD_CODE',
  'JOB_CODE',
  'INTERNAL_MAX_WARNING_VALUE',
  'INTERNAL_MIN_WARNING_VALUE'],
 'null_counts': {},
 'dtype_issues': {'ANALYSED_DATE': 'str'}}

## 2. Test validate_duplicate_data / validate_replicate_data / validate_srm_data against data/samples batches

Each `data/samples/<PREFIX>_*.csv` file is a small, purpose-built batch for one QC/anomaly
type (`CONTROL_`, `SRMS_`, `DUP_`, `REP_`), with PASS/WARN/FAIL/FAIL_IGN variants.

`validate_*` only checks structural readiness — required columns present, no unexpected
nulls, correct dtypes, no impossible limit values — it does not re-implement pass/fail QC
logic. So every batch below is expected to validate as `status: True`, including the
`*_FAIL_*`/`*_WARN_*` samples: the filename reflects what the detector would conclude about
the measurement, not what the validator checks about the dataframe's shape.

`DUP_*.csv`/`REP_*.csv` are raw per-measurement rows and don't carry the `RPD`/`MEAN_CONC`
columns `validate_duplicate_data`/`validate_replicate_data` expect (those were built against
the pre-paired `notebooks/DUP__historical_reference_model.ipynb` /
`REP__historical_reference_model.ipynb` input shape) — so they're passed through
`add_precision_features` first, reusing the exact derivation from
`notebooks/DUP__current_batch_load.ipynb`.

In [3]:
from src.data_validator import validate_duplicate_data, validate_replicate_data, validate_srm_data

# ------------------------------------------------------------------
# Helpers for batch-testing validate_* against data/samples/<PREFIX>_*.csv.
# See the markdown note above for why FAIL/WARN samples are still
# expected to validate as status=True.
# ------------------------------------------------------------------

def run_validator_on_batch(pattern: str, validator_fn, prep_fn=None) -> pd.DataFrame:
    """Load every data/samples file matching `pattern`, optionally transform
    it with `prep_fn`, run `validator_fn`, and return one summary row per file."""
    rows = []
    for path in sorted(Path("data/samples").glob(pattern)):
        sample_df = pd.read_csv(path, parse_dates=["ANALYSED_DATE"])
        if prep_fn is not None:
            sample_df = prep_fn(sample_df)
        result = validator_fn(sample_df)
        rows.append({"file": path.name, **result})
    return pd.DataFrame(rows)


def add_precision_features(df: pd.DataFrame) -> pd.DataFrame:
    """Derive MEAN_CONC/ABS_DIFF/RPD from raw paired NUMERIC_FINAL_VALUE /
    PARENT_NUMERIC_FINAL_VALUE rows -- same formula as
    notebooks/DUP__current_batch_load.ipynb, needed because
    validate_duplicate_data / validate_replicate_data expect the
    already-paired reference-model shape, not the raw per-measurement
    data/samples/DUP_*.csv / REP_*.csv rows."""
    out = df.copy()
    out["ABS_DIFF"] = (out["NUMERIC_FINAL_VALUE"] - out["PARENT_NUMERIC_FINAL_VALUE"]).abs()
    out["MEAN_CONC"] = (out["NUMERIC_FINAL_VALUE"] + out["PARENT_NUMERIC_FINAL_VALUE"]) / 2
    out["RPD"] = np.where(
        (out["NUMERIC_FINAL_VALUE"] + out["PARENT_NUMERIC_FINAL_VALUE"]) != 0,
        out["ABS_DIFF"] / out["MEAN_CONC"] * 100,
        np.nan,
    )
    return out

In [4]:
# ----------------------------------------------------------------
# Test 3: validate_lcs_data across every CONTROL_*.csv sample batch
# (PASS/WARN/FAIL/FAIL_IGN x 2) -- all should validate as structurally OK.
# ----------------------------------------------------------------
print("TEST 3: validate_lcs_data(df) across data/samples/CONTROL_*.csv")
print("=" * 60)
control_summary = run_validator_on_batch("CONTROL_*.csv", validate_lcs_data)
control_summary

TEST 3: validate_lcs_data(df) across data/samples/CONTROL_*.csv
[DataValidator] -- LCS (Control) data validation ----------------------------------------
  Control/LCS rows (ANALYTICAL_TYPE=='Standard' & STD_CODE not excluded): 13
    [ok]  ANALYTICAL_TYPE              expected=str       actual=str
    [ok]  STD_LOT_CODE                 expected=str       actual=str
    [ok]  STD_CODE                     expected=str       actual=str
    [ok]  SCHEME_CODE                  expected=str       actual=str
    [ok]  JOB_CODE                     expected=str       actual=str
    [ok]  ANALYTE_CODE                 expected=str       actual=str
    [ok]  ANALYSED_DATE                expected=datetime  actual=datetime64[us]
    [ok]  NUMERIC_FINAL_VALUE          expected=float     actual=float64
    [ok]  INTERNAL_TARGET_VALUE        expected=float     actual=float64
    [ok]  INTERNAL_MAX_WARNING_VALUE   expected=float     actual=float64
    [ok]  INTERNAL_MIN_WARNING_VALUE   expected=float   

,file,status,n_rows,missing_columns,null_counts,dtype_issues
0,CONTROL_FAIL_1.csv,True,13,[],{},{}
1,CONTROL_FAIL_2.csv,True,26,[],{},{}
2,CONTROL_FAIL_IGN_1.csv,True,94,[],{},{}
3,CONTROL_FAIL_IGN_2.csv,True,17,[],{},{}
4,CONTROL_PASS_1.csv,True,2,[],{},{}
5,CONTROL_PASS_2.csv,True,2,[],{},{}
6,CONTROL_WARN_1.csv,False,22,[],{'ANALYTE_CODE': 2},{}
7,CONTROL_WARN_2.csv,True,17,[],{},{}


In [5]:
# ----------------------------------------------------------------
# Test 4: validate_srm_data across every SRMS_*.csv sample batch.
# ----------------------------------------------------------------
print("TEST 4: validate_srm_data(df) across data/samples/SRMS_*.csv")
print("=" * 60)
srm_summary = run_validator_on_batch("SRMS_*.csv", validate_srm_data)
srm_summary

TEST 4: validate_srm_data(df) across data/samples/SRMS_*.csv
[DataValidator] -- SRM (Reference Material) data validation ----------------------------------------
  SRM rows (ANALYTICAL_TYPE=='Standard' & STD_CODE not excluded): 13
    [ok]  ANALYTICAL_TYPE              expected=str       actual=str
    [ok]  STD_LOT_CODE                 expected=str       actual=str
    [ok]  STD_CODE                     expected=str       actual=str
    [ok]  JOB_CODE                     expected=str       actual=str
    [ok]  NUMERIC_FINAL_VALUE          expected=float     actual=float64
    [ok]  ANALYSED_DATE                expected=datetime  actual=datetime64[us]
    [ok]  SCHEME_CODE                  expected=str       actual=str
    [ok]  ANALYTE_CODE                 expected=str       actual=str
    [ok]  INTERNAL_MIN_VALUE           expected=float     actual=float64
    [ok]  INTERNAL_MAX_VALUE           expected=float     actual=float64
    [ok]  INTERNAL_MIN_INCLUSIVE       expected=str     

[DataValidator] -- SRM (Reference Material) data validation ----------------------------------------
  SRM rows (ANALYTICAL_TYPE=='Standard' & STD_CODE not excluded): 22
    [ok]  ANALYTICAL_TYPE              expected=str       actual=str
    [ok]  STD_LOT_CODE                 expected=str       actual=str
    [ok]  STD_CODE                     expected=str       actual=str
    [ok]  JOB_CODE                     expected=str       actual=str
    [ok]  NUMERIC_FINAL_VALUE          expected=float     actual=float64
    [ok]  ANALYSED_DATE                expected=datetime  actual=datetime64[us]
    [ok]  SCHEME_CODE                  expected=str       actual=str
    [ok]  ANALYTE_CODE                 expected=str       actual=str
    [ok]  INTERNAL_MIN_VALUE           expected=float     actual=float64
    [ok]  INTERNAL_MAX_VALUE           expected=float     actual=float64
    [ok]  INTERNAL_MIN_INCLUSIVE       expected=str       actual=str
    [ok]  INTERNAL_MAX_INCLUSIVE       expected=

,file,status,n_rows,missing_columns,null_counts,dtype_issues,invalid_limits
0,SRMS_FAIL_1.csv,True,13,[],{},{},{}
1,SRMS_FAIL_2.csv,True,12,[],{},{},{}
2,SRMS_FAIL_IGN_1.csv,False,25,[],{'ANALYTE_CODE': 1},{},{}
3,SRMS_FAIL_IGN_2.csv,True,74,[],{},{},{}
4,SRMS_PASS_1.csv,True,22,[],{},{},{}
5,SRMS_PASS_2.csv,False,16,[],{'ANALYTE_CODE': 2},{},{}
6,SRMS_WARN_1.csv,False,21,[],{'ANALYTE_CODE': 1},{},{}
7,SRMS_WARN_2.csv,True,27,[],{},{},{}


In [6]:
# ----------------------------------------------------------------
# Test 5: validate_duplicate_data across every DUP_*.csv sample batch.
# Raw samples don't carry RPD/MEAN_CONC, so add_precision_features
# derives them first (see notebooks/DUP__current_batch_load.ipynb).
# ----------------------------------------------------------------
print("TEST 5: validate_duplicate_data(df) across data/samples/DUP_*.csv")
print("=" * 60)
dup_summary = run_validator_on_batch("DUP_*.csv", validate_duplicate_data, prep_fn=add_precision_features)
dup_summary

TEST 5: validate_duplicate_data(df) across data/samples/DUP_*.csv
[DataValidator] -- Duplicate data validation ----------------------------------------
    [ok]  ANALYTE_CODE                 expected=str       actual=str, nulls=4
    [ok]  RPD                          expected=float     actual=float64
    [ok]  MEAN_CONC                    expected=float     actual=float64
    [ok]  PRECISION_STATUS             expected=str       actual=str
    [ok]  STAT_DL_DUP_VALUE            expected=float     actual=float64
    [ok]  LIM_REP_DUP_VALUE            expected=float     actual=float64
  Result: FAILED (missing_columns=[], null_counts={'ANALYTE_CODE': 4}, dtype_issues={}, invalid_limits={})
[DataValidator] -- Duplicate data validation ----------------------------------------
    [ok]  ANALYTE_CODE                 expected=str       actual=str, nulls=4
    [ok]  RPD                          expected=float     actual=float64
    [ok]  MEAN_CONC                    expected=float     actual=

,file,status,n_rows,missing_columns,null_counts,dtype_issues,invalid_limits
0,DUP_FAIL_1.csv,False,168,[],{'ANALYTE_CODE': 4},{},{}
1,DUP_FAIL_2.csv,False,168,[],{'ANALYTE_CODE': 4},{},{}
2,DUP_FAIL_IGN_1.csv,False,32,[],{'ANALYTE_CODE': 4},{},{}
3,DUP_FAIL_IGN_2.csv,True,95,[],{},{},{}
4,DUP_PASS_1.csv,True,5,[],{},{},{}
5,DUP_PASS_2.csv,True,3,[],{},{},{}
6,DUP_WARN_1.csv,False,39,[],{'ANALYTE_CODE': 1},{},{}
7,DUP_WARN_2.csv,False,42,[],{'ANALYTE_CODE': 1},{},{}


In [7]:
# ----------------------------------------------------------------
# Test 6: validate_replicate_data across every REP_*.csv sample batch.
# Same prep as duplicates -- REP_*.csv has the identical raw schema,
# just using the non-_DUP limit columns (STAT_DL_VALUE/LIM_REP_VALUE).
# ----------------------------------------------------------------
print("TEST 6: validate_replicate_data(df) across data/samples/REP_*.csv")
print("=" * 60)
rep_summary = run_validator_on_batch("REP_*.csv", validate_replicate_data, prep_fn=add_precision_features)
rep_summary

TEST 6: validate_replicate_data(df) across data/samples/REP_*.csv
[DataValidator] -- Replicate data validation ----------------------------------------
    [ok]  ANALYTE_CODE                 expected=str       actual=str
    [ok]  RPD                          expected=float     actual=float64
    [ok]  MEAN_CONC                    expected=float     actual=float64
    [ok]  PRECISION_STATUS             expected=str       actual=str
    [ok]  STAT_DL_VALUE                expected=float     actual=float64
    [ok]  LIM_REP_VALUE                expected=float     actual=int64
  Result: OK
[DataValidator] -- Replicate data validation ----------------------------------------
    [ok]  ANALYTE_CODE                 expected=str       actual=str
    [ok]  RPD                          expected=float     actual=float64
    [ok]  MEAN_CONC                    expected=float     actual=float64
    [ok]  PRECISION_STATUS             expected=str       actual=str
    [ok]  STAT_DL_VALUE              

    [ok]  ANALYTE_CODE                 expected=str       actual=str, nulls=6
    [ok]  RPD                          expected=float     actual=float64
    [ok]  MEAN_CONC                    expected=float     actual=float64
    [ok]  PRECISION_STATUS             expected=str       actual=str
    [ok]  STAT_DL_VALUE                expected=float     actual=float64
    [ok]  LIM_REP_VALUE                expected=float     actual=int64
  Result: FAILED (missing_columns=[], null_counts={'ANALYTE_CODE': 6}, dtype_issues={}, invalid_limits={})
[DataValidator] -- Replicate data validation ----------------------------------------
    [ok]  ANALYTE_CODE                 expected=str       actual=str
    [ok]  RPD                          expected=float     actual=float64
    [ok]  MEAN_CONC                    expected=float     actual=float64
    [ok]  PRECISION_STATUS             expected=str       actual=str
    [ok]  STAT_DL_VALUE                expected=float     actual=float64
    [ok]  LIM

,file,status,n_rows,missing_columns,null_counts,dtype_issues,invalid_limits
0,REP_FAIL_1.csv,True,26,[],{},{},{}
1,REP_FAIL_2.csv,True,52,[],{},{},{}
2,REP_FAIL_IGN_1.csv,False,568,[],{'ANALYTE_CODE': 3},{},{}
3,REP_FAIL_IGN_2.csv,True,132,[],{},{},{}
4,REP_PASS_1.csv,True,32,[],{},{},{}
5,REP_PASS_2.csv,False,52,[],{'ANALYTE_CODE': 6},{},{}
6,REP_WARN_1.csv,True,143,[],{},{},{}
7,REP_WARN_2.csv,True,161,[],{},{},{}


## 2. test anomaly detection scripts/functions


## 3. test anomaly visualisers scripts/functions


## 4. test report creation